# Data Cleaning - Club Piscine MMM

This notebook extracts and cleans data from the client's Excel files for use in the Marketing Mix Model.

**Files processed:**
1. Historical sales by store and by division for 2023-2024-2025 - Actual sales data (revenue + units)
2. Budget 2023, 2024 & 2025 - Media spend by channel
3. Recap Tableau Medias 2025 - Campaign performance metrics
4. Calendrier Fiscal - Fiscal calendar reference
5. Preroll 2025 - Google Ads breakdown (Display vs Video) for FY2025

**Note:** Blank cells represent missing data (client confirmed) - we preserve these as NaN without imputation.

**UPDATE (2026-02-09):** Switched dependent variable from quote requests (soumissions) to actual sales data.
- Sales data provides 36 months of complete data (FY2023-FY2025) vs 19 months with quotes
- Both units and revenue available; revenue ($) used as primary target for ROI optimization
- Fiscal year already correctly assigned in sales file (Année fiscale column)

---

## Media Channel Grouping Strategy (v2)

To optimize for limited historical data (~36 monthly observations), we consolidate media channels into **7 strategic groups**:

| # | Group | Budget Source Channels | Rationale |
|---|-------|------------------------|----------|
| 1 | **Television** | TELEVISION | Major investment, long carryover effect |
| 2 | **Radio** | RADIO + RADIO NUMÉRIQUE | Audio channels |
| 3 | **Panneaux** | PANNEAUX + PANNEAUX ET AFFICHAGES NUMÉRIQUES | Outdoor exposure |
| 4 | **Social_Media** | FACEBOOK + INSTAGRAM + PINTEREST + TIKTOK | Social platforms (already combined) |
| 5 | **Preroll** | PREROLL PREMIUM + YOUTUBE (video) | Video content |
| 6 | **Banniere_Web** | PREMIUM DISPLAY + BANNIÈRES WEB + GOOGLE ADS + LAPRESSE + CONTENU DE MARQUE (image) | Display/image content |
| 7 | **Circulaire_Digitale** | CIRCULAIRE DIGITALE | Digital flyers |

**Excluded channels:** PROGRAMMATIQUE, AUDIO ET PODCAST, ENVOIS POSTAUX

In [19]:
# Cell 1: Imports and Setup
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Paths - adjust based on where this notebook is run
project_root = Path().cwd().parent if Path().cwd().name == 'notebooks' else Path().cwd()
raw_path = project_root / 'data' / 'raw'
processed_path = project_root / 'data' / 'processed'

# Create processed directory if it doesn't exist
processed_path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")
print(f"Raw data path: {raw_path}")
print(f"Processed data path: {processed_path}")

# List raw files
print("\nRaw files found:")
for f in raw_path.glob('*'):
    print(f"  - {f.name}")

Project root: /Users/raoul/Dev/busa693-clubpiscine
Raw data path: /Users/raoul/Dev/busa693-clubpiscine/data/raw
Processed data path: /Users/raoul/Dev/busa693-clubpiscine/data/processed

Raw files found:
  - CalendrierFiscal.xlsx
  - Budget 2025 - 21 août.xlsx
  - Budget_2023_.xlsx
  - Rapport de soumissions 2025.xlsx
  - ~$Preroll  2025.xlsx
  - Budget 2023 .xlsx
  - Historical sales by store and by division for 2023-2024-2025.xlsx
  - Rapport de soumissions 2024.xlsx
  - Recap_Tableau_Medias_2025.xlsx
  - Preroll  2025.xlsx
  - Budget 2024 - REEL au 5 novembre.xlsx
  - ~$Budget 2023 .xlsx


---
## 1. Sales Data (Historical Sales by Store and Division) - FY2023 to FY2025

Extracts and aggregates actual sales data from the client's store-level weekly file.

**Source file:** `Historical sales by store and by division for 2023-2024-2025.xlsx`
- Sheet: "Ventes cumulatives par magasin"
- Row 0: Meta-header (product group names) — skipped
- Row 1: Actual column headers — used as `header=1`
- Data starts at row 2 (0-indexed)
- 6,336 rows of weekly data across 42 stores
- Fiscal year already assigned in `Année fiscale` column

**Fiscal Year Definition:**
- FY2023: Oct 31, 2022 → Oct 30, 2023
- FY2024: Nov 6, 2023 → Oct 28, 2024
- FY2025: Nov 4, 2024 → Oct 27, 2025

**6 Product Categories (all equal):**
- HT: Piscines Hors Terre (above-ground pools) — units + revenue
- CR: Piscines Creusées (in-ground pools) — units + revenue
- SP: Spas — units + revenue
- ME&GA: Meubles & Gazebo (furniture & gazebo combined) — revenue only (single column `$-ME & $-GA` in raw data)
- FI: Fitness — revenue only
- BQ: BBQ — revenue only

**Aggregation:** Weekly store-level → Monthly company-level (36 rows)

**Note:** Replaces previous soumissions (quote requests) data which only had 19 usable months.

In [20]:
# Cell 2: Extract and Aggregate Sales Data

def extract_sales_data(raw_path):
    """
    Extract and aggregate sales data from Historical sales file.
    
    Aggregates weekly store-level data (6,336 rows, 42 stores)
    to monthly company-level totals (36 rows).
    
    The sales file has:
    - Row 0: Meta-header (product group labels) — skipped
    - Row 1: Actual column headers
    - Data from row 2 onwards
    - Fiscal year already assigned in 'Année fiscale' column
    
    6 Product Categories (all equal):
    - HT, CR, SP (units + revenue), ME&GA, FI, BQ (revenue only)
    - ME&GA uses the combined '$-ME & $-GA' column from raw data
    """
    df = pd.read_excel(
        raw_path / 'Historical sales by store and by division for 2023-2024-2025.xlsx',
        sheet_name='Ventes cumulatives par magasin',
        header=1  # Row 0 is meta-header, Row 1 is actual headers
    )
    
    print(f"  Raw data loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"  Stores: {df['Code magasin'].nunique()}")
    print(f"  Date range: {df['Commence le'].min().date()} to {df['Commence le'].max().date()}")
    print(f"  Fiscal years: {sorted(df['Année fiscale'].unique())}")
    
    # Aggregate across all stores, group by fiscal year + calendar month
    # Use '$-ME & $-GA' combined column (not separate $-ME and $-GA)
    monthly = df.groupby(['Année fiscale', 'Month']).agg({
        'U-HT': 'sum',           # Above-ground pools - units
        '$-HT': 'sum',           # Above-ground pools - revenue
        'U-CR': 'sum',           # In-ground pools - units
        '$-CR': 'sum',           # In-ground pools - revenue
        'U-SP': 'sum',           # Spas - units
        '$-SP': 'sum',           # Spas - revenue
        '$-ME & $-GA': 'sum',    # Meubles & Gazebo combined - revenue
        '$-FI': 'sum',           # Fitness - revenue
        '$-BQ': 'sum'            # BBQ - revenue
    }).reset_index()
    
    # Rename to project conventions
    monthly = monthly.rename(columns={
        'Année fiscale': 'year',       # This IS the fiscal year
        'Month': 'month_num',           # Calendar month (1-12)
        'U-HT': 'piscines_hors_terre_units',
        '$-HT': 'piscines_hors_terre_revenue',
        'U-CR': 'piscines_creusees_units',
        '$-CR': 'piscines_creusees_revenue',
        'U-SP': 'spas_units',
        '$-SP': 'spas_revenue',
        '$-ME & $-GA': 'meubles_gazebo_revenue',
        '$-FI': 'fitness_revenue',
        '$-BQ': 'bbq_revenue'
    })
    
    # Add total revenue across all 6 categories
    monthly['total_all_revenue'] = (
        monthly['piscines_hors_terre_revenue'] +
        monthly['piscines_creusees_revenue'] +
        monthly['spas_revenue'] +
        monthly['meubles_gazebo_revenue'] +
        monthly['fitness_revenue'] +
        monthly['bbq_revenue']
    )
    
    # Total units (only HT, CR, SP have unit data)
    monthly['total_units'] = (
        monthly['piscines_hors_terre_units'] +
        monthly['piscines_creusees_units'] +
        monthly['spas_units']
    )
    
    # Create date column for time series
    # month_num is calendar month; derive calendar year from FY + month
    # Nov-Dec of FY X → calendar year X-1; Jan-Oct of FY X → calendar year X
    monthly['calendar_year'] = monthly.apply(
        lambda r: int(r['year']) - 1 if r['month_num'] >= 11 else int(r['year']),
        axis=1
    )
    monthly['date'] = pd.to_datetime(
        monthly['calendar_year'].astype(str) + '-' + monthly['month_num'].astype(str) + '-01'
    )
    monthly = monthly.sort_values('date').reset_index(drop=True)
    
    # Add month name
    month_names = {
        1: 'Janvier', 2: 'Février', 3: 'Mars', 4: 'Avril',
        5: 'Mai', 6: 'Juin', 7: 'Juillet', 8: 'Août',
        9: 'Septembre', 10: 'Octobre', 11: 'Novembre', 12: 'Décembre'
    }
    monthly['month'] = monthly['month_num'].map(month_names)
    
    return monthly

# Extract sales data
sales_data = extract_sales_data(raw_path)

print("\n" + "=" * 60)
print("SALES DATA - EXTRACTED & AGGREGATED")
print("=" * 60)
print(f"Shape: {sales_data.shape}")
print(f"Date range: {sales_data['date'].min().date()} to {sales_data['date'].max().date()}")
print(f"\nFiscal years: {sorted(sales_data['year'].unique())}")
print(f"Months per FY:")
for fy in sorted(sales_data['year'].unique()):
    n = len(sales_data[sales_data['year'] == fy])
    months = sorted(sales_data[sales_data['year'] == fy]['month_num'].tolist())
    print(f"  FY{fy}: {n} months → {months}")

print(f"\nMissing values: {sales_data.isna().sum().sum()}")
print(f"\nRevenue summary by fiscal year:")
for fy in sorted(sales_data['year'].unique()):
    fy_data = sales_data[sales_data['year'] == fy]
    total = fy_data['total_all_revenue'].sum()
    print(f"  FY{fy}: ${total:,.0f}")

print(f"\n6 Product Categories (all equal):")
print(f"  HT: Piscines Hors Terre (Above-Ground Pools)")
print(f"  CR: Piscines Creusées (In-Ground Pools)")
print(f"  SP: Spas")
print(f"  ME&GA: Meubles & Gazebo (Furniture & Gazebo)")
print(f"  FI: Fitness")
print(f"  BQ: BBQ")

print(f"\nSample data (first 5 rows):")
print(sales_data[['year', 'month_num', 'month', 'piscines_hors_terre_revenue', 
                   'piscines_creusees_revenue', 'spas_revenue', 'meubles_gazebo_revenue',
                   'fitness_revenue', 'bbq_revenue', 'total_all_revenue']].head().to_string())

  Raw data loaded: 6,336 rows × 16 columns
  Stores: 42
  Date range: 2022-10-31 to 2025-10-27
  Fiscal years: [np.int64(2023), np.int64(2024), np.int64(2025)]

SALES DATA - EXTRACTED & AGGREGATED
Shape: (36, 16)
Date range: 2022-11-01 to 2025-10-01

Fiscal years: [np.int64(2023), np.int64(2024), np.int64(2025)]
Months per FY:
  FY2023: 12 months → [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
  FY2024: 12 months → [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
  FY2025: 12 months → [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

Missing values: 0

Revenue summary by fiscal year:
  FY2023: $173,631,029
  FY2024: $163,984,500
  FY2025: $174,796,073

6 Product Categories (all equal):
  HT: Piscines Hors Terre (Above-Ground Pools)
  CR: Piscines Creusées (In-Ground Pools)
  SP: Spas
  ME&GA: Meubles & Gazebo (Furniture & Gazebo)
  FI: Fitness
  BQ: BBQ

Sample data (first 5 rows):
   year  month_num     month  piscines_hors_terre_revenue  piscines_creusees_revenue  spas_revenue  meubles_gazebo_revenue  fitne

---
## 2. Budget Media Spend - With 7-Channel Grouping

Extracts monthly media spend from Budget 2023, 2024, and 2025 files and groups into 7 strategic channels for MMM:

1. **Television** - TV spend
2. **Radio** - Radio + Radio Numérique
3. **Panneaux** - Panneaux + Panneaux et Affichages Numériques
4. **Social_Media** - Facebook (Promo+Produit) + Instagram (Promo+Produit) + Pinterest + TikTok
5. **Preroll** - Preroll Premium + Youtube (video content)
6. **Banniere_Web** - Premium Display + Bannières Web (Google Ads) + La Presse + Contenu de Marque (image content)
7. **Circulaire_Digitale** - Digital flyers

**Note:** Budget 2023 uses slightly different naming (e.g., 'GOOGLE' vs 'GOOGLE ADS', 'LAPRESSE+' vs 'LAPRESSE')

**Excluded:** PROGRAMMATIQUE, AUDIO ET PODCAST, ENVOIS POSTAUX

In [21]:
# Cell 3: Clean Budget Media Spend with 7-Channel Grouping (2023, 2024, 2025)

def clean_budget_grouped(file_path, year):
    """
    Clean the budget file and group channels into 7 strategic categories.
    
    Updated channel grouping (2026-02-09):
    1. Television: TELEVISION
    2. Radio: RADIO + RADIO NUMÉRIQUE
    3. Panneaux: PANNEAUX + PANNEAUX ET AFFICHAGES NUMÉRIQUES
    4. Social_Media: FACEBOOK + INSTAGRAM + PINTEREST + TIKTOK
    5. Preroll: PREROLL PREMIUM + YOUTUBE (video content)
    6. Banniere_Web: PREMIUM DISPLAY + BANNIÈRES WEB (Google Ads) + LAPRESSE + CONTENU DE MARQUE (image)
    7. Circulaire_Digitale: CIRCULAIRE DIGITALE / CIRCULAIRE DIGITAL
    
    Excluded: PROGRAMMATIQUE, AUDIO ET PODCAST, ENVOIS POSTAUX, GOOGLE SHOPPING
    
    FIX (2026-02-09): 
    - Month detection takes LAST matching column (event sub-columns like "Évènement/juillet" 
      appear before the actual monthly total "VENTE 1/JUILLET")
    - Google parent row excluded; sub-rows used instead to avoid Google Shopping contamination
    """
    df_raw = pd.read_excel(file_path, sheet_name=0, header=None)
    
    # Define channel grouping mapping
    CHANNEL_GROUPS = {
        # 1. Television
        'TELEVISION': 'Television',
        
        # 2. Radio
        'RADIO': 'Radio',
        'RADIO NUMÉRIQUE': 'Radio',
        
        # 3. Panneaux (outdoor)
        'PANNEAUX': 'Panneaux',
        'PANNEAUX ET AFFICHAGES NUMÉRIQUES': 'Panneaux',
        
        # 4. Social Media
        'FACEBOOK': 'Social_Media',
        'FACEBOOK + INSTAGRAM (PROMO)': 'Social_Media',
        'FACEBOOK + INSTAGRAM (PRODUIT)': 'Social_Media',
        'PINTEREST': 'Social_Media',
        'TIKTOK': 'Social_Media',
        
        # 5. Preroll (video content)
        'PREROLL - PREMIUM': 'Preroll',
        'PREROLL - YOUTUBE': 'Preroll',         # Google YouTube preroll sub-row (2023)
        
        # 6. Banniere_Web (display/image content)
        'BANNIÈRES WEB - PREMIUM': 'Banniere_Web',
        'BANNIERES WEB - PREMIUM': 'Banniere_Web',  # Without accent variant
        'BANNIERES WEB': 'Banniere_Web',
        'BANNIÈRES WEB': 'Banniere_Web',             # Google Display sub-row (2023/2024)             # Google Display sub-row (2023)
        # 'GOOGLE ADS' parent row → EXCLUDED (sub-rows used instead)
        # 'GOOGLE DISPLAY + PREROLL' → EXCLUDED (split via Preroll file)
        # 'RECHERCHE DE MOTS' → EXCLUDED (search spend not in model)
        'LAPRESSE+': 'Banniere_Web',                 # 2023 format
        'LAPRESSE (LP+, PREROLL, DISPLAY)': 'Banniere_Web',  # 2024/2025 format
        'CONTENU DE MARQUE': 'Banniere_Web',
        
        # 7. Circulaire Digitale
        'CIRCULAIRE DIGITAL': 'Circulaire_Digitale',   # 2023 format
        'CIRCULAIRE DIGITALE': 'Circulaire_Digitale',   # 2024/2025 format
    }
    
    # Channels to exclude completely
    EXCLUDE_PATTERNS = [
        'PROGRAMMATIQUE',
        'AUDIO ET PODCAST',
        'ENVOIS POSTAUX',
        'COMMANDITES',
        'GOOGLE SHOPPING',     # Dropped from analysis
        'RECHERCHE DE MOTS',   # FIX: Search spend excluded
    ]
    
    # Skip patterns - headers, totals, subtotals, sub-components
    SKIP_EXACT = ['FR', 'EN', 'ENG', 'TRADITIONNEL', 'NUMÉRIQUE', 'NUMERIQUE', 'AUTRES']
    SKIP_CONTAINS = [
        'TOTAL', 'DIFFÉRENCE', '% VS', 
        'SEMAINE', 'CAMPAGNE', 'MEDIA', 'COOP', 'PRODUCTION', 'RÉSERVE',
        'CONTINGENCE', 'CIRCULAIRE PAPIER',
        # Sub-components (avoid double-counting)
        'VIDEO ( PORTÉE', 'PERFORMANCE ( SOUMISSIONS',
        'GOOGLE DISPLAY + PREROLL',     # 2025 parent — split via Preroll file
    ]
    
    # Month mapping - fiscal year starts in November
    months_info = [
        ('NOVEMBRE', 11), ('DECEMBRE', 12), ('JANVIER', 1), ('FEVRIER', 2),
        ('MARS', 3), ('AVRIL', 4), ('MAI', 5), ('JUIN', 6),
        ('JUILLET', 7), ('AOUT', 8), ('SEPTEMBRE', 9), ('OCTOBRE', 10)
    ]
    
    # Find column indices for each month in row 6
    # FIX: Take the LAST matching column, not the first. Some months have 
    # event sub-columns (e.g., "Évènement/juillet" at col 39) before the 
    # actual monthly total column ("VENTE 1/JUILLET" at col 40).
    row6 = df_raw.iloc[6, :].tolist()
    month_cols = {}
    for i, val in enumerate(row6[:70]):
        if pd.notna(val):
            val_upper = str(val).upper().strip()
            for month_name, month_num in months_info:
                if val_upper == month_name:  # Always overwrite to take LAST match
                    month_cols[month_name] = (i, month_num)
                    break
    
    print(f"  Found {len(month_cols)} months for {year}")
    
    # Data rows to process (rows 11-42 in Excel = indices 10-41)
    data_rows = list(range(10, 42))
    
    media_data = []
    
    for row_idx in data_rows:
        media_name = df_raw.iloc[row_idx, 3]  # Column D
        
        if pd.isna(media_name) or not str(media_name).strip():
            continue
        
        media_name_str = str(media_name).strip()
        media_name_upper = media_name_str.upper()
        
        # Skip exact matches
        if media_name_upper in SKIP_EXACT:
            continue
        
        # Skip if contains certain patterns (but allow BANNIÈRES WEB - PREMIUM)
        skip_this = False
        for skip in SKIP_CONTAINS:
            if skip in media_name_upper:
                skip_this = True
                break
        if skip_this:
            continue
        
        # Skip excluded channels
        if any(excl.upper() in media_name_upper for excl in EXCLUDE_PATTERNS):
            continue
        
        # Skip plain BANNIÈRES WEB parent row (not PREMIUM) to avoid double-counting
        # BUT allow Google's 'Bannières web' sub-row (col0='DISPLAY') → Banniere_Web
        if media_name_upper == 'BANNIÈRES WEB':
            col0_val = str(df_raw.iloc[row_idx, 0]).strip().upper() if pd.notna(df_raw.iloc[row_idx, 0]) else ''
            if col0_val == 'DISPLAY':
                pass  # This is Google Display sub-row — don't skip, map to Banniere_Web below
            else:
                continue
        
        # Skip GOOGLE ADS parent row — use sub-rows instead (2024)
        if media_name_upper == 'GOOGLE ADS':
            continue
        
        # Skip GOOGLE parent row — use sub-rows instead to avoid Google Shopping contamination
        if media_name_upper == 'GOOGLE':
            continue
        
        # Determine channel group
        channel_group = None
        for pattern, group in CHANNEL_GROUPS.items():
            if pattern == media_name_upper or pattern in media_name_upper:
                channel_group = group
                break
        
        # Skip if no group mapping found
        if channel_group is None:
            continue
        
        # Extract spend for each month
        for month_name, (col_idx, month_num) in month_cols.items():
            spend = df_raw.iloc[row_idx, col_idx]
            spend_value = pd.to_numeric(spend, errors='coerce')
            
            if pd.notna(spend_value) and spend_value != 0:
                media_data.append({
                    'year': year,
                    'month': month_name,
                    'month_num': month_num,
                    'channel_group': channel_group,
                    'spend': spend_value
                })
    
    if not media_data:
        print(f"      range used: rows 10-41, reading col 3")
        for ri in [10, 13, 16, 22, 29, 35]:
            if ri < len(df_raw):
                print(f"      Row {ri}: col3={df_raw.iloc[ri, 3]}")
    df_clean = pd.DataFrame(media_data)
    
    if df_clean.empty:
        return df_clean
    
    # Aggregate by year, month, and channel group
    df_agg = df_clean.groupby(['year', 'month', 'month_num', 'channel_group'], 
                               as_index=False)['spend'].sum()
    
    return df_agg

# Process ALL THREE budget files
print("Processing Budget 2023...")
budget_2023 = clean_budget_grouped(raw_path / 'Budget_2023_.xlsx', 2023)

print("Processing Budget 2024...")
budget_2024 = clean_budget_grouped(raw_path / 'Budget 2024 - REEL au 5 novembre.xlsx', 2024)

print("Processing Budget 2025...")
budget_2025 = clean_budget_grouped(raw_path / 'Budget 2025 - 21 août.xlsx', 2025)

# Combine ALL years
budget_combined = pd.concat([budget_2023, budget_2024, budget_2025], ignore_index=True)

# =====================================================================
# FIX (2026-02-23): Integrate Preroll 2025 file
# =====================================================================
# Budget 2025 'GOOGLE DISPLAY + PREROLL' parent row is now excluded.
# The Preroll file breaks Google costs into Display_Cost and Video_Cost.
# Display_Cost -> Banniere_Web, Video_Cost -> Preroll
# Search_Cost, Shopping_Cost, PerfMax_Cost -> EXCLUDED

def load_preroll_breakdown(file_path):
    df = pd.read_excel(file_path, header=2)
    df.columns = ['Month', 'Currency', 'Search_Cost', 'Display_Cost',
                  'Shopping_Cost', 'Video_Cost', 'PerfMax_Cost']
    for col in ['Display_Cost', 'Video_Cost']:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    month_map = {
        'january': 1, 'february': 2, 'march': 3, 'april': 4,
        'may': 5, 'june': 6, 'july': 7, 'august': 8,
        'september': 9, 'october': 10, 'november': 11, 'december': 12
    }
    month_name_fr = {
        1: 'JANVIER', 2: 'FEVRIER', 3: 'MARS', 4: 'AVRIL',
        5: 'MAI', 6: 'JUIN', 7: 'JUILLET', 8: 'AOUT',
        9: 'SEPTEMBRE', 10: 'OCTOBRE', 11: 'NOVEMBRE', 12: 'DECEMBRE'
    }
    records = []
    for _, row in df.iterrows():
        month_str = str(row['Month']).strip()
        if not month_str or month_str == 'nan':
            continue
        parts = month_str.split()
        if len(parts) < 2:
            continue
        month_num = month_map.get(parts[0].lower())
        cal_year = int(parts[1])
        if month_num is None:
            continue
        fiscal_year = cal_year + 1 if month_num >= 11 else cal_year
        if row['Display_Cost'] > 0:
            records.append({
                'year': fiscal_year, 'month': month_name_fr[month_num],
                'month_num': month_num, 'channel_group': 'Banniere_Web',
                'spend': row['Display_Cost']
            })
        if row['Video_Cost'] > 0:
            records.append({
                'year': fiscal_year, 'month': month_name_fr[month_num],
                'month_num': month_num, 'channel_group': 'Preroll',
                'spend': row['Video_Cost']
            })
    return pd.DataFrame(records)

preroll_path = raw_path / 'Preroll  2025.xlsx'
if preroll_path.exists():
    preroll_data = load_preroll_breakdown(preroll_path)
    preroll_fy25 = preroll_data[preroll_data['year'] == 2025]
    display_total = preroll_fy25[preroll_fy25['channel_group']=='Banniere_Web']['spend'].sum()
    video_total = preroll_fy25[preroll_fy25['channel_group']=='Preroll']['spend'].sum()
    print(f"\nPreroll file applied for FY2025:")
    print(f"  Google Display -> Banniere_Web: ${display_total:,.2f}")
    print(f"  Google Video   -> Preroll:      ${video_total:,.2f}")
    budget_combined = pd.concat([budget_combined, preroll_fy25], ignore_index=True)
    budget_combined = budget_combined.groupby(
        ['year', 'month', 'month_num', 'channel_group'], as_index=False
    )['spend'].sum()
else:
    print(f"\nPreroll file not found at {preroll_path}")


# Create pivot for display
month_order = ['NOVEMBRE', 'DECEMBRE', 'JANVIER', 'FEVRIER', 'MARS', 'AVRIL', 
               'MAI', 'JUIN', 'JUILLET', 'AOUT', 'SEPTEMBRE', 'OCTOBRE']

print("\n" + "=" * 70)
print("BUDGET - 7 CHANNEL GROUPS")
print("=" * 70)

for year in [2023, 2024, 2025]:
    print(f"\n--- FISCAL YEAR {year} ---")
    year_data = budget_combined[budget_combined['year'] == year]
    if not year_data.empty:
        pivot = year_data.pivot_table(index='channel_group', columns='month', 
                                       values='spend', aggfunc='sum', fill_value=0)
        pivot = pivot[[m for m in month_order if m in pivot.columns]]
        print(pivot.round(0).to_string())
        print(f"\nTotal spend {year}: ${year_data['spend'].sum():,.0f}")
        
# Show channel totals across ALL years
print("\n" + "=" * 70)
print("TOTAL SPEND BY CHANNEL (2023-2025 Combined)")
print("=" * 70)
channel_totals = budget_combined.groupby('channel_group')['spend'].sum().sort_values(ascending=False)
for ch, total in channel_totals.items():
    pct = total / channel_totals.sum() * 100
    print(f"  {ch}: ${total:,.0f} ({pct:.1f}%)")
print(f"\n  TOTAL: ${channel_totals.sum():,.0f}")

Processing Budget 2023...
  Found 12 months for 2023
Processing Budget 2024...
  Found 12 months for 2024
Processing Budget 2025...
  Found 12 months for 2025

Preroll file applied for FY2025:
  Google Display -> Banniere_Web: $99,910.09
  Google Video   -> Preroll:      $89,745.58

BUDGET - 7 CHANNEL GROUPS

--- FISCAL YEAR 2023 ---
month                NOVEMBRE  DECEMBRE  JANVIER   FEVRIER      MARS     AVRIL       MAI      JUIN   JUILLET     AOUT  SEPTEMBRE  OCTOBRE
channel_group                                                                                                                            
Banniere_Web          16668.0    4924.0  12548.0   22497.0   20535.0   43637.0   41802.0   14028.0   18089.0  12876.0    22853.0   6301.0
Circulaire_Digitale       0.0   10883.0   5006.0   10012.0       0.0   23895.0   26946.0   21407.0   19139.0  18836.0        0.0      0.0
Panneaux                  0.0       0.0      0.0       0.0       0.0       0.0   22922.0    4090.0   78150.0   8

In [22]:
# Cell 3b: Create wide-format budget for merging with sales data

# Pivot to wide format: one row per year-month, columns for each channel
budget_wide = budget_combined.pivot_table(
    index=['year', 'month', 'month_num'],
    columns='channel_group',
    values='spend',
    aggfunc='sum',
    fill_value=0
).reset_index()

# Flatten column names
budget_wide.columns = ['year', 'month', 'month_num'] + \
    [f'spend_{col.lower().replace(" ", "_")}' for col in budget_wide.columns[3:]]

# Add total spend column
spend_cols = [c for c in budget_wide.columns if c.startswith('spend_')]
budget_wide['spend_total'] = budget_wide[spend_cols].sum(axis=1)

print("Wide-format budget created:")
print(f"  Shape: {budget_wide.shape}")
print(f"  Years: {sorted(budget_wide['year'].unique())}")
print(f"  Columns: {budget_wide.columns.tolist()}")

Wide-format budget created:
  Shape: (36, 11)
  Years: [np.int64(2023), np.int64(2024), np.int64(2025)]
  Columns: ['year', 'month', 'month_num', 'spend_banniere_web', 'spend_circulaire_digitale', 'spend_panneaux', 'spend_preroll', 'spend_radio', 'spend_social_media', 'spend_television', 'spend_total']


---
## 3. Tableau Medias 2025 - Campaign Performance

Extracts campaign-level metrics from the MASTER-TOTAL sheet.

**Columns extracted:**
- B (1): Date début - Campaign start date
- C (2): Date fin - Campaign end date  
- D (3): Média - Media type
- F (5): Station / Support - Media partner/platform
- L (11): Coût total ($ NET) - Total cost
- T (19): Nb occasions (RÉEL) - Actual number of spots
- U (20): Impressions totales (RÉEL) - Actual impressions
- V (21): PEB (RÉEL) - Actual GRPs
- AB (27): Vues complétées - Completed views
- AC (28): Taux de vues - View rate
- AD (29): Clics (RÉEL) - Actual clicks
- AE (30): Taux de clics - Click rate

**Note:** We also map to 7-channel groups for consistency

In [23]:
# Cell 4: Clean Tableau Medias 2025

def clean_tableau_medias(file_path):
    """
    Clean the Tableau Medias file and map to 7 channel groups.
    
    Updated channel groups (2026-02-09):
    Television, Radio, Panneaux, Social_Media, Preroll, Banniere_Web, Circulaire_Digitale
    """
    df_raw = pd.read_excel(file_path, sheet_name='MASTER-TOTAL', header=0)
    
    # Extract columns by index
    cols_to_extract = {
        1: 'date_debut',
        2: 'date_fin', 
        3: 'media_type',
        5: 'support',
        11: 'cost_net',
        19: 'occasions_reel',
        20: 'impressions_reel',
        21: 'peb_reel',
        27: 'vues_completees',
        28: 'taux_vues',
        29: 'clics_reel',
        30: 'taux_clics'
    }
    
    df_clean = df_raw.iloc[:, list(cols_to_extract.keys())].copy()
    df_clean.columns = list(cols_to_extract.values())
    
    # Remove empty rows
    df_clean = df_clean.dropna(how='all')
    df_clean = df_clean.dropna(subset=['date_debut', 'date_fin'], how='all')
    
    # Convert date columns
    df_clean['date_debut'] = pd.to_datetime(df_clean['date_debut'], errors='coerce')
    df_clean['date_fin'] = pd.to_datetime(df_clean['date_fin'], errors='coerce')
    
    # Convert numeric columns
    numeric_cols = ['cost_net', 'occasions_reel', 'impressions_reel', 'peb_reel',
                    'vues_completees', 'taux_vues', 'clics_reel', 'taux_clics']
    for col in numeric_cols:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    
    # Map to 7 channel groups based on media_type and support
    def map_to_channel_group(row):
        media_type = str(row['media_type']).upper() if pd.notna(row['media_type']) else ''
        support = str(row['support']).upper() if pd.notna(row['support']) else ''
        
        if media_type == 'TÉLÉVISION':
            return 'Television'
        elif media_type == 'RADIO':
            return 'Radio'
        elif media_type == 'AFFICHAGE':
            return 'Panneaux'
        elif media_type == 'NUMÉRIQUE':
            # Digital channels - use support to differentiate
            if 'CIRCULAIRE' in support or 'FLIPP' in support:
                return 'Circulaire_Digitale'
            elif any(x in support for x in ['FACEBOOK', 'INSTAGRAM', 'PINTEREST', 'TIKTOK']):
                return 'Social_Media'
            elif any(x in support for x in ['PREROLL', 'YOUTUBE']):
                return 'Preroll'
            else:
                # Google Ads, La Presse, display, content = Banniere_Web
                return 'Banniere_Web'
        else:
            return 'Other'
    
    df_clean['channel_group'] = df_clean.apply(map_to_channel_group, axis=1)
    
    # Add derived columns
    df_clean['year'] = df_clean['date_debut'].dt.year
    df_clean['month'] = df_clean['date_debut'].dt.month
    
    # Calculate CPM
    mask = (df_clean['impressions_reel'] > 0) & df_clean['cost_net'].notna()
    df_clean.loc[mask, 'cpm_calculated'] = (
        df_clean.loc[mask, 'cost_net'] / df_clean.loc[mask, 'impressions_reel']
    ) * 1000
    
    df_clean = df_clean.reset_index(drop=True)
    
    return df_clean

# Process the file
tableau_medias = clean_tableau_medias(raw_path / 'Recap_Tableau_Medias_2025.xlsx')

print("=" * 60)
print("TABLEAU MEDIAS 2025 - Campaign Performance")
print("=" * 60)
print(f"\nShape: {tableau_medias.shape}")
print(f"Date range: {tableau_medias['date_debut'].min()} to {tableau_medias['date_fin'].max()}")

print("\n--- By 7 Channel Groups ---")
summary = tableau_medias.groupby('channel_group').agg({
    'cost_net': 'sum',
    'impressions_reel': 'sum',
    'clics_reel': 'sum'
}).round(0)
print(summary.to_string())

TABLEAU MEDIAS 2025 - Campaign Performance

Shape: (383, 16)
Date range: 1970-01-01 00:00:00.000000006 to 2025-10-31 00:00:00

--- By 7 Channel Groups ---
                      cost_net  impressions_reel  clics_reel
channel_group                                               
Banniere_Web          500542.0        52476771.0    495103.0
Circulaire_Digitale   183900.0         7494436.0     32390.0
Panneaux              181491.0       236384489.0         0.0
Preroll               296320.0        11041521.0     29670.0
Radio                 799103.0        57016900.0         0.0
Social_Media          214864.0        40848093.0    203791.0
Television           1042994.0        82131818.0         0.0


---
## 4. Calendrier Fiscal (Fiscal Calendar)

Reference table for fiscal calendar mapping. The client's fiscal year starts in November.

In [24]:
# Cell 5: Clean Calendrier Fiscal

def clean_calendrier_fiscal(file_path):
    """
    Clean the Calendrier Fiscal file.
    """
    df_raw = pd.read_excel(file_path, sheet_name='CalendrierFiscal', header=0)
    
    columns_to_keep = [
        'Date', 'Année', 'Mois', 'Nom Mois', 'Jour de la semaine',
        'Année fiscale', 'Trimestre', 'Semaine fiscale', 'Formule',
        'Semaine débutant le', 'Ordre du mois fiscal', 'Ordre semaine',
        'MoisFiscal', 'AnnéeNUM', 'Date début semaine'
    ]
    
    existing_cols = [col for col in columns_to_keep if col in df_raw.columns]
    df_clean = df_raw[existing_cols].copy()
    
    # Convert Date column
    if 'Date' in df_clean.columns:
        df_clean['Date'] = pd.to_datetime(df_clean['Date'], errors='coerce')
    
    # Remove rows where Date is NaN
    df_clean = df_clean.dropna(subset=['Date'])
    
    return df_clean

# Process the file
calendrier_fiscal = clean_calendrier_fiscal(raw_path / 'CalendrierFiscal.xlsx')

print("=" * 60)
print("CALENDRIER FISCAL")
print("=" * 60)
print(f"\nShape: {calendrier_fiscal.shape}")
print(f"Date range: {calendrier_fiscal['Date'].min()} to {calendrier_fiscal['Date'].max()}")
print(f"Fiscal years: {sorted(calendrier_fiscal['Année fiscale'].dropna().unique())}")

CALENDRIER FISCAL

Shape: (1890, 15)
Date range: 2021-11-01 00:00:00 to 2027-01-03 00:00:00
Fiscal years: ['2022 fiscale', '2023 fiscale', '2024 fiscale', '2025 fiscale', '2026 fiscale', '2027 fiscale']


---
## 5. Save All Cleaned Datasets

In [25]:
# Cell 6: Save all cleaned dataframes

# Save sales data (replaces soumissions_quotes)
sales_data.to_csv(processed_path / 'sales_data.csv', index=False)
sales_data.to_pickle(processed_path / 'sales_data.pkl')

# Save budget - long format (for flexibility)
budget_combined.to_csv(processed_path / 'budget_media_spend.csv', index=False)
budget_combined.to_pickle(processed_path / 'budget_media_spend.pkl')

# Save budget - wide format (for merging)
budget_wide.to_csv(processed_path / 'budget_media_spend_wide.csv', index=False)
budget_wide.to_pickle(processed_path / 'budget_media_spend_wide.pkl')

# Save tableau medias
tableau_medias.to_csv(processed_path / 'tableau_medias_performance.csv', index=False)
tableau_medias.to_pickle(processed_path / 'tableau_medias_performance.pkl')

# Save calendrier fiscal
calendrier_fiscal.to_csv(processed_path / 'calendrier_fiscal.csv', index=False)
calendrier_fiscal.to_pickle(processed_path / 'calendrier_fiscal.pkl')

print("=" * 60)
print("FILES SAVED")
print("=" * 60)
print(f"\nSaved to: {processed_path}")
print("\nFiles created:")
for f in sorted(processed_path.glob('*')):
    size_kb = f.stat().st_size / 1024
    print(f"  - {f.name} ({size_kb:.1f} KB)")

FILES SAVED

Saved to: /Users/raoul/Dev/busa693-clubpiscine/data/processed

Files created:
  - bayesian_mmm_roas.csv (0.8 KB)
  - bayesian_vs_ridge_comparison.csv (0.6 KB)
  - budget_media_spend.csv (6.4 KB)
  - budget_media_spend.pkl (5.9 KB)
  - budget_media_spend_wide.csv (3.6 KB)
  - budget_media_spend_wide.pkl (4.2 KB)
  - calendrier_fiscal.csv (341.1 KB)
  - calendrier_fiscal.pkl (175.3 KB)
  - causal_model_params.json (2.4 KB)
  - external_weather.csv (2.6 KB)
  - external_weather.pkl (5.4 KB)
  - media_effectiveness_results.csv (8.7 KB)
  - mmm_executive_summary.csv (0.7 KB)
  - mmm_final_output.json (1.0 KB)
  - mmm_optimization_results.csv (0.6 KB)
  - mmm_roi_results.csv (1.7 KB)
  - mmm_scenario_analysis.csv (0.6 KB)
  - model_B_frisch_waugh.csv (0.7 KB)
  - model_C_channel_results.csv (0.9 KB)
  - model_C_params.json (2.9 KB)
  - optimal_transformation_params.json (2.2 KB)
  - robustness_summary.csv (0.2 KB)
  - sales_data.csv (4.3 KB)
  - sales_data.pkl (6.4 KB)
  - sales

---
## 5b. Merge Sales Data with Media Spend

Creates the merged dataset for downstream analysis (EDA in NB03, weather in NB04, modeling in NB05/NB06).

- Inner join on `(year, month_num)` — both use fiscal year directly.
- Adds `fiscal_month_pos` for fiscal-year-ordered plotting.
- Saves as `sales_spend_merged.pkl` / `.csv`.

In [26]:
# Cell 6b: Merge Sales Data with Media Spend

# Merge sales with wide-format budget on (year, month_num)
budget_merge_cols = ['year', 'month_num'] + [c for c in budget_wide.columns if c.startswith('spend_')]
merged = sales_data.merge(budget_wide[budget_merge_cols], on=['year', 'month_num'], how='inner')

# Add fiscal month position for plotting (Nov=1, Dec=2, ..., Oct=12)
merged['fiscal_month_pos'] = merged['month_num'].apply(
    lambda m: m - 10 if m >= 11 else m + 2
)

# Save merged dataset
merged.to_csv(processed_path / 'sales_spend_merged.csv', index=False)
merged.to_pickle(processed_path / 'sales_spend_merged.pkl')

print("=" * 60)
print("MERGED DATASET: Sales + Media Spend")
print("=" * 60)
print(f"Shape: {merged.shape}")
print(f"Fiscal years: {sorted(merged['year'].unique())}")
print(f"Months per FY:")
for fy in sorted(merged['year'].unique()):
    print(f"  FY{fy}: {len(merged[merged['year'] == fy])} months")

spend_cols = [c for c in merged.columns if c.startswith('spend_')]
print(f"\nSpend columns: {spend_cols}")
print(f"\nSaved: sales_spend_merged.pkl / .csv")

MERGED DATASET: Sales + Media Spend
Shape: (36, 25)
Fiscal years: [np.int64(2023), np.int64(2024), np.int64(2025)]
Months per FY:
  FY2023: 12 months
  FY2024: 12 months
  FY2025: 12 months

Spend columns: ['spend_banniere_web', 'spend_circulaire_digitale', 'spend_panneaux', 'spend_preroll', 'spend_radio', 'spend_social_media', 'spend_television', 'spend_total']

Saved: sales_spend_merged.pkl / .csv


---
## 7. Data Quality Summary

In [27]:
# Cell 8: Data Quality Summary

print("=" * 70)
print("DATA QUALITY SUMMARY")
print("=" * 70)

datasets = {
    'Sales Data (Revenue + Units)': sales_data,
    'Budget (Media Spend - Long)': budget_combined,
    'Budget (Media Spend - Wide)': budget_wide,
    'Sales + Spend Merged': merged,
    'Tableau Medias': tableau_medias,
    'Calendrier Fiscal': calendrier_fiscal
}

for name, df in datasets.items():
    print(f"\n{'-' * 40}")
    print(f"{name}")
    print(f"{'-' * 40}")
    print(f"  Rows: {len(df):,}")
    print(f"  Columns: {len(df.columns)}")
    
    total_cells = df.size
    missing_cells = df.isna().sum().sum()
    missing_pct = (missing_cells / total_cells) * 100
    print(f"  Missing values: {missing_cells:,} ({missing_pct:.1f}%)")

print("\n" + "=" * 70)
print("7 CHANNEL GROUPS FOR MMM")
print("=" * 70)
print("""
  1. Television          - TV spend
  2. Radio               - Radio + Radio Numérique
  3. Panneaux            - Panneaux + Panneaux et Affichages Numériques
  4. Social_Media        - Facebook + Instagram + Pinterest + TikTok
  5. Preroll             - Preroll Premium + Youtube (video)
  6. Banniere_Web        - Premium Display + Bannières Web (Google Ads) + LaPresse + Contenu de Marque (image)
  7. Circulaire_Digitale - Digital flyers
  
  EXCLUDED: Programmatic, Audio/Podcast, Postal, Google Shopping
""")

print("=" * 70)
print("NOTE: Missing values (NaN) preserved as per client instructions.")
print("      Sales data has 0 missing values across 36 months.")
print("      sales_spend_merged.pkl is the primary input for NB03 (EDA).")
print("=" * 70)

DATA QUALITY SUMMARY

----------------------------------------
Sales Data (Revenue + Units)
----------------------------------------
  Rows: 36
  Columns: 16
  Missing values: 0 (0.0%)

----------------------------------------
Budget (Media Spend - Long)
----------------------------------------
  Rows: 171
  Columns: 5
  Missing values: 0 (0.0%)

----------------------------------------
Budget (Media Spend - Wide)
----------------------------------------
  Rows: 36
  Columns: 11
  Missing values: 0 (0.0%)

----------------------------------------
Sales + Spend Merged
----------------------------------------
  Rows: 36
  Columns: 25
  Missing values: 0 (0.0%)

----------------------------------------
Tableau Medias
----------------------------------------
  Rows: 383
  Columns: 16
  Missing values: 1,763 (28.8%)

----------------------------------------
Calendrier Fiscal
----------------------------------------
  Rows: 1,890
  Columns: 15
  Missing values: 0 (0.0%)

7 CHANNEL GROUPS FOR